# 01 — Dataset Generation

Generates exactly **300** balanced triangle-inequality True/False prompts and
splits them into **train (150) / eval (75) / analysis (75)**.

Outputs written to `my_work/data/`:
- `prompts_triangle.jsonl` — full 300-prompt dataset
- `splits/train_triangle.jsonl`
- `splits/eval.jsonl`
- `splits/analysis.jsonl`

Run this notebook once.  No GPU required.

## 0 — Environment Setup

In [2]:
import os
import sys
from pathlib import Path

IN_RUNPOD = os.environ.get("RUNPOD_POD_ID") is not None

def _find_repo_root():
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "circuit_tracer" / "__init__.py").is_file():
            return directory
    workspace = Path("/workspace")
    if workspace.is_dir():
        for child in workspace.iterdir():
            if child.is_dir() and (child / "circuit_tracer" / "__init__.py").is_file():
                return child
    repo_override = os.environ.get("CT_REPO_DIR")
    if repo_override:
        override_path = Path(repo_override).expanduser().resolve()
        if (override_path / "circuit_tracer" / "__init__.py").is_file():
            return override_path
    return None

_root = _find_repo_root()
if _root is not None:
    if str(_root) not in sys.path:
        sys.path.insert(0, str(_root))
    # Add my_work to path so utils/ imports work
    _my_work = _root / "my_work"
    if str(_my_work) not in sys.path:
        sys.path.insert(0, str(_my_work))
    print(f"Repo root: {_root}")
    print(f"my_work  : {_my_work}")
else:
    print("WARNING: could not locate circuit_tracer repo. Set CT_REPO_DIR.")

try:
    from dotenv import load_dotenv
    if _root is not None:
        _env_file = _root / ".env"
        if _env_file.is_file():
            load_dotenv(_env_file)
            print(f"Loaded {_env_file}")
except ImportError:
    pass

# Derive MY_WORK path used by all output cells
MY_WORK = _my_work if _root else Path(".").resolve()
print(f"MY_WORK  : {MY_WORK}")

Repo root: /Users/Lacus/Documents/ELTE/Thesis/circuit-tracer-main
my_work  : /Users/Lacus/Documents/ELTE/Thesis/circuit-tracer-main/my_work
MY_WORK  : /Users/Lacus/Documents/ELTE/Thesis/circuit-tracer-main/my_work


## 1 — Generate 300 prompts

In [3]:
import json
import sys
from collections import Counter

from utils.prompt_generator import generate_triangle_prompts, split_dataset, TOKEN_TRUE, TOKEN_FALSE

SEED = 42
N_PROMPTS = 300

prompts = generate_triangle_prompts(n=N_PROMPTS, seed=SEED)

print(f"Generated {len(prompts)} prompts")
print(f"  True : {sum(1 for p in prompts if p['label'])}")
print(f"  False: {sum(1 for p in prompts if not p['label'])}")
print(f"  Degenerate triples: {sum(1 for p in prompts if not p['triangle_valid'])}")
print()
print("Template distribution:")

template_cnt = Counter(p['template_id'] for p in prompts)
def sort_key(t):
    # Sort by type name, then value as string (for consistent ordering)
    return (str(type(t[0])), str(t[0]))
for tid, cnt in sorted(template_cnt.items(), key=sort_key):
    print(f"  template {tid}: {cnt}")
print()
print("First 3 examples:")
for p in prompts[:3]:
    print(f"  [{p['prompt_id']}] label={p['label']} tmpl={p['template_id']}")
    print(f"    {p['prompt'][:90]}")

Generated 300 prompts
  True : 150
  False: 150
  Degenerate triples: 30

Template distribution:
  template 1: 64
  template 2: 69
  template 3: 63
  template 4: 74
  template 5: 10
  template 6: 10
  template 7: 5
  template 8: 5

First 3 examples:
  [tri_001] label=True tmpl=4
    Given sides 3 and 7, 8 is an allowable third side for a triangle. Answer:
  [tri_002] label=False tmpl=4
    Given sides 3 and 5, 6 is not an allowable third side for a triangle. Answer:
  [tri_003] label=False tmpl=3
    For a triangle with sides 4 and 9, the third side cannot be 10. Answer:


## 2 — Sanity checks

In [4]:
# Verify label_token matches label
for p in prompts:
    expected_tok = TOKEN_TRUE if p['label'] else TOKEN_FALSE
    assert p['label_token'] == expected_tok, f"{p['prompt_id']}: token mismatch"

# Verify degenerate fraction ~10%
n_degen = sum(1 for p in prompts if not p['triangle_valid'])
degen_frac = n_degen / len(prompts)
assert 0.08 <= degen_frac <= 0.12, f"Degenerate fraction {degen_frac:.2%} outside [8%,12%]"

# Verify no duplicate prompt_ids
ids = [p['prompt_id'] for p in prompts]
assert len(ids) == len(set(ids)), "Duplicate prompt_ids found"

# Verify required schema fields
required_fields = {'prompt_id','prompt','label','label_token','sides',
                   'triangle_valid','template_id','claim_direction'}
for p in prompts:
    missing = required_fields - set(p.keys())
    assert not missing, f"{p['prompt_id']}: missing fields {missing}"

print(f"All sanity checks passed. Degenerate fraction: {degen_frac:.1%}")

All sanity checks passed. Degenerate fraction: 10.0%


## 3 — Save full dataset

In [5]:
data_dir = MY_WORK / "data"
data_dir.mkdir(parents=True, exist_ok=True)

full_path = data_dir / "prompts_triangle.jsonl"
with open(full_path, "w", encoding="utf-8") as f:
    for p in prompts:
        f.write(json.dumps(p) + "\n")

print(f"Saved {len(prompts)} prompts to {full_path}")

Saved 300 prompts to /Users/Lacus/Documents/ELTE/Thesis/circuit-tracer-main/my_work/data/prompts_triangle.jsonl


## 4 — Split into train / eval / analysis

Sizes: **150 train · 75 eval · 75 analysis** (stratified by label).

In [6]:
splits = split_dataset(
    prompts,
    train_frac=0.5,
    eval_frac=0.25,
    analysis_frac=0.25,
    seed=SEED,
)

splits_dir = data_dir / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

split_file_map = {
    "train":    "train_triangle.jsonl",
    "eval":     "eval.jsonl",
    "analysis": "analysis.jsonl",
}

for split_name, filename in split_file_map.items():
    entries = splits[split_name]
    out_path = splits_dir / filename
    with open(out_path, "w", encoding="utf-8") as f:
        for e in entries:
            f.write(json.dumps(e) + "\n")
    n_t = sum(1 for e in entries if e['label'])
    n_f = sum(1 for e in entries if not e['label'])
    print(f"  {split_name:<10} → {out_path.name}  n={len(entries)}  (T={n_t} F={n_f})")

# Verify no overlap between train and eval/analysis
train_ids = {e['prompt_id'] for e in splits['train']}
eval_ids   = {e['prompt_id'] for e in splits['eval']}
ana_ids    = {e['prompt_id'] for e in splits['analysis']}
assert not (train_ids & eval_ids),     "Overlap between train and eval!"
assert not (train_ids & ana_ids),      "Overlap between train and analysis!"
assert not (eval_ids  & ana_ids),      "Overlap between eval and analysis!"
print("\nNo overlap between splits. Dataset ready.")

  train      → train_triangle.jsonl  n=150  (T=75 F=75)
  eval       → eval.jsonl  n=76  (T=38 F=38)
  analysis   → analysis.jsonl  n=74  (T=37 F=37)

No overlap between splits. Dataset ready.


## 5 — Preview analysis split

A quick look at what the attribution loop will process.

In [7]:
analysis = splits['analysis']
print(f"Analysis split: {len(analysis)} prompts")
print(f"  True : {sum(1 for e in analysis if e['label'])}")
print(f"  False: {sum(1 for e in analysis if not e['label'])}")
print(f"  Degenerate: {sum(1 for e in analysis if not e['triangle_valid'])}")
print()
print("Sample prompts:")
for e in analysis[:4]:
    print(f"  [{e['prompt_id']}] label={e['label']} tmpl={e['template_id']}")
    print(f"    {e['prompt'][:90]}")

Analysis split: 74 prompts
  True : 37
  False: 37
  Degenerate: 8

Sample prompts:
  [tri_212] label=True tmpl=4
    Given sides 10 and 10, 1 is an allowable third side for a triangle. Answer:
  [tri_046] label=True tmpl=1
    There can be a triangle with side lengths 2, 2, and 3. Answer:
  [tri_249] label=False tmpl=3
    For a triangle with sides 3 and 7, the third side cannot be 8. Answer:
  [tri_137] label=False tmpl=3
    For a triangle with sides 4 and 4, the third side cannot be 5. Answer:
